In [19]:
pip install --upgrade websockets


In [20]:
pip install aiohttp


In [ ]:
import asyncio
import aiohttp
import websockets
import json
import time
import requests

# =============================================================================
# USER CONFIGURATION - STRATEGY & RISK
# =============================================================================
USER_EMAIL = "shreyansh2004knp@gmail.com"
USER_PASSWORD = "QuantBootcamp1$55"
VIRTUAL_SUBACCOUNT_NAME = "virtual_subaccount_shreyansh2004knp@gmail.com"

# --- STRATEGY SETTINGS ---
PREFERRED_EXECUTION_EXCHANGE = "okx"
USDT_TO_SPEND_PER_TRADE = 50.0
USDT_RESERVE = 100.0
MINIMUM_ETH_ORDER_SIZE = 0.001
MAX_NEGATIVE_USDT_BALANCE = -150.0

# =============================================================================
# SYSTEM CONFIGURATION - DO NOT MODIFY
# =============================================================================
ACCOUNT_MAP = {
    "okx": "skyfallokxsub2",
    "bybit": "skyfallbybitsub2",
    "kucoinspot": "skyfallkucoinsub2"
}
ACCOUNT_NAME = ACCOUNT_MAP[PREFERRED_EXECUTION_EXCHANGE]
WEBSOCKET_URL = "wss://quant-bootcamp-api.goquant.io/ws/v1/virtual-subaccount"
AUTH_URL = "https://quant-bootcamp-api.goquant.io/auth/v2/validate_user"
TRADING_CONFIG = {
    "exchange_name": PREFERRED_EXECUTION_EXCHANGE,
    "account_name": ACCOUNT_NAME,
    "base": "ETH",
    "quote": "USDT",
    "duration": 15,
    "algorithm_type": "market_edge"
}

def get_access_token(email, password):
    print("LOG: Entering get_access_token")
    try:
        response = requests.post(AUTH_URL, json={"email": email, "password": password})
        if response.status_code == 200:
            data = response.json()
            if data.get("type") == "success":
                return data["data"]["access_token"]
        print(f"❌ Authentication failed: {response.text}")
        return None
    except Exception as e:
        print(f"❌ Authentication error: {e}")
        return None

class TradingBot:
    def __init__(self):
        self.ws = None
        self.access_token = None
        self.total_available_usdt = 0.0
        self.total_eth_position = 0.0
        self.portfolio_state = {}
        print("LOG: TradingBot initialized with Unified Portfolio view.")

    async def get_current_price(self):
        print("LOG: Entering get_current_price")
        url = "https://api.coinbase.com/v2/prices/ETH-USDT/spot"
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get(url) as resp:
                    if resp.status == 200:
                        data = await resp.json()
                        if "data" in data and "amount" in data["data"]:
                            price = float(data["data"]["amount"])
                            print(f"ℹ️ Current ETH/USDT price (from Coinbase): {price}")
                            return price
                    print(f"⚠️ Error fetching from Coinbase API. Status: {resp.status}, Response: {await resp.text()}")
                    return None
        except Exception as e:
            print(f"⚠️ An unexpected exception occurred in get_current_price: {e}")
            return None

    async def run(self):
        print("LOG: Starting TradingBot run cycle")
        print("=" * 40)
        print(f"Preferred Exchange: {ACCOUNT_NAME}")
        print(f"Strategy: Spend ${USDT_TO_SPEND_PER_TRADE} per trade, keep ${USDT_RESERVE} total reserve.")
        print(f"RISK LIMIT: Bot will stop if TOTAL USDT balance falls below ${MAX_NEGATIVE_USDT_BALANCE}.")
        print("=" * 40)

        try:
            if not await self.connect(): return
            await self.subscribe_channels()

            while True:
                await self.check_balance()

                if self.total_available_usdt < MAX_NEGATIVE_USDT_BALANCE:
                    print(f"\n🚨 CRITICAL RISK LIMIT BREACHED 🚨\nTotal USDT ({self.total_available_usdt:.2f}) is below the safety threshold of ${MAX_NEGATIVE_USDT_BALANCE:.2f}.\nShutting down.")
                    break

                # This function will now handle both long and short positions to bring the bot to neutral.
                position_was_flattened = await self.flatten_position_if_needed()

                if not position_was_flattened and self.total_available_usdt > USDT_RESERVE:
                    print("\n--- Evaluating Buy Condition ---")
                    print(f"✅ Total capital sufficient (${self.total_available_usdt:.2f} > ${USDT_RESERVE:.2f}). Proceeding with buy.")
                    buy_algo_id = await self.place_order(side="buy")
                    if buy_algo_id:
                        await self.wait_for_algo_completion(buy_algo_id)
                        print("LOG: Waiting 3 seconds for balance to settle after buy...")
                        await asyncio.sleep(3)
                else:
                    print("\n--- Skipping Buy Condition ---")
                    reason = "Just flattened a position." if position_was_flattened else f"Insufficient total capital. Available: ${self.total_available_usdt:.2f}, Reserve required: ${USDT_RESERVE:.2f}"
                    print(f"ℹ️ Reason: {reason}")

                print("\n--- Cycle complete. Waiting for 30 seconds before next cycle. ---")
                await asyncio.sleep(30)

        except asyncio.CancelledError:
            print("\nLOG: Task was cancelled. Shutting down.")
        except Exception as e:
            print(f"⚠️ An critical error occurred in the main run loop: {e}")
        finally:
            if self.ws and self.ws.state == websockets.protocol.State.OPEN:
                await self.ws.close()
                print("🔴 WebSocket closed")

    async def flatten_position_if_needed(self):
        """Checks for any significant long or short position and closes it to return to neutral."""
        print("\n--- Checking for Existing Total Positions (Risk Management) ---")

        # --- NEW LOGIC: Handle both LONG and SHORT positions ---

        # Case 1: We have a LONG position that needs to be sold.
        if self.total_eth_position >= MINIMUM_ETH_ORDER_SIZE:
            print(f"✅ Found total LONG position of {self.total_eth_position} ETH. Placing SELL order to flatten.")
            sell_algo_id = await self.place_order(side="sell", quantity=self.total_eth_position)
            if sell_algo_id:
                await self.wait_for_algo_completion(sell_algo_id)
                print("LOG: Waiting 3 seconds for balance to settle after flattening...")
                await asyncio.sleep(3)
                return True # Signal that an action was taken

        # Case 2: We have a SHORT position that needs to be bought back.
        elif self.total_eth_position <= -MINIMUM_ETH_ORDER_SIZE:
            # We need to buy back the absolute amount we are short
            quantity_to_buy_back = abs(self.total_eth_position)
            print(f"✅ Found total SHORT position of {self.total_eth_position} ETH. Placing BUY order to flatten.")
            buy_algo_id = await self.place_order(side="buy", quantity=quantity_to_buy_back)
            if buy_algo_id:
                await self.wait_for_algo_completion(buy_algo_id)
                print("LOG: Waiting 3 seconds for balance to settle after flattening...")
                await asyncio.sleep(3)
                return True # Signal that an action was taken

        # Case 3: No significant position exists.
        else:
            print("ℹ️ No significant total position found to flatten.")

        return False # Signal that no action was taken

    async def connect(self):
        self.access_token = get_access_token(USER_EMAIL, USER_PASSWORD)
        if not self.access_token: return False
        headers = {"Authorization": f"Bearer {self.access_token}", "Virtual-Subaccount-Name": VIRTUAL_SUBACCOUNT_NAME}
        try:
            self.ws = await websockets.connect(WEBSOCKET_URL, additional_headers=headers)
            print("✅ WebSocket connected successfully.")
            return True
        except Exception as e:
            print(f"❌ WebSocket connection failed: {e}")
            return False

    async def subscribe_channels(self):
        for channel in ["algorithms", "orders"]:
            await self.ws.send(json.dumps({"op": "subscribe", "channel": channel}))
            await self.ws.recv()

    async def check_balance(self):
        print("LOG: Requesting portfolio balance...")
        await self.ws.send(json.dumps({"op": "virtual_subaccount_balance"}))
        while True:
            msg = json.loads(await self.ws.recv())
            if "virtual_subaccount_balance" in msg:
                self.log_full_balance(msg)
                self.update_portfolio_state(msg)
                return

    def log_full_balance(self, balance_msg):
        print("\n--- Full JSON Balance File Received ---")
        print(json.dumps(balance_msg, indent=2))
        print("---------------------------------------\n")

    def update_portfolio_state(self, balance_msg):
        self.portfolio_state = {}
        temp_total_usdt = 0.0
        temp_total_eth = 0.0
        for account in balance_msg.get("virtual_subaccount_balance", []):
            exchange_name = account.get("exchange_name")
            usdt_assets = account.get("assets", {}).get("USDT", {})
            eth_assets = account.get("assets", {}).get("ETH", {})
            usdt_avail = float(usdt_assets.get("available", 0.0))
            eth_avail = float(eth_assets.get("available", 0.0))
            temp_total_usdt += usdt_avail
            temp_total_eth += eth_avail
            self.portfolio_state[exchange_name] = {"USDT": usdt_avail, "ETH": eth_avail}

        self.total_available_usdt = temp_total_usdt
        self.total_eth_position = temp_total_eth
        print("--- Portfolio State Updated ---")
        for exch, bals in self.portfolio_state.items():
            print(f"  - {exch}: USDT: {bals['USDT']:.2f}, ETH: {bals['ETH']:.4f}")
        print(f"  -----------------------------")
        print(f"  TOTALS: USDT: {self.total_available_usdt:.2f}, ETH: {self.total_eth_position:.4f}")
        print("-----------------------------")

    async def place_order(self, side, quantity=None):
        print(f"LOG: Entering place_order for a '{side}' order.")
        # For strategic buys, calculate quantity based on risk capital.
        # For flattening buys/sells, quantity is passed in directly.
        if side == "buy" and quantity is None:
            price = await self.get_current_price()
            if not price: return None
            usdt_value_of_trade = USDT_TO_SPEND_PER_TRADE * 2.0
            quantity = usdt_value_of_trade / price
            print(f"LOG: Calculated strategic buy quantity: {quantity:.6f} ETH for ${usdt_value_of_trade:.2f}")

        if quantity < MINIMUM_ETH_ORDER_SIZE:
            print(f"❌ Aborting {side}: Calculated quantity {quantity:.8f} is below minimum of {MINIMUM_ETH_ORDER_SIZE}.")
            return None

        payload = {"op": "place", **TRADING_CONFIG, "side": side, "quantity": round(quantity, 6)}
        print(f"🚀 Placing {side} order: {payload}")
        await self.ws.send(json.dumps(payload))

        while True:
            msg = json.loads(await self.ws.recv())
            if "algorithm_place_response" in msg:
                resp = msg["algorithm_place_response"]
                if resp.get("status") == "success":
                    client_algo_id = resp["client_algo_id"]
                    print(f"✅ Order accepted by API. Algo ID: {client_algo_id}")
                    return client_algo_id
                else:
                    print(f"❌ Order failed on placement: {resp}")
                    return None
            elif msg.get("status") == "error":
                 print(f"❌ API Error on placement: {msg.get('message')}")
                 return None

    async def wait_for_algo_completion(self, client_algo_id):
        print(f"⏳ Waiting for Algo {client_algo_id} to complete...")
        terminal_states = ["completed", "error", "rejected", "canceled"]
        while True:
            msg = json.loads(await self.ws.recv())
            if "algorithms" in msg:
                algo = msg["algorithms"]
                if str(algo.get("client_algo_id")) == str(client_algo_id):
                    state = algo.get("state")
                    print(f"📊 Algo {client_algo_id} update: State is now '{state}'")
                    if state in terminal_states:
                        print(f"✅ Algo {client_algo_id} has finished.")
                        return
            elif "orders" in msg:
                 order = msg["orders"]
                 if str(order.get("client_algo_id")) == str(client_algo_id):
                     print(f"📑 Order update for Algo {client_algo_id}: {order.get('state')}, Reason: {order.get('oems_order_update', {}).get('rejection_reason')}")

async def main():
    print("GoQuant Trading Script")
    print("Make sure to configure your credentials at the top of this script!\n")
    bot = TradingBot()
    await bot.run()
    print("LOG: Script finished")

if __name__ == "__main__":
    await main()

Streaming output truncated to the last 5000 lines.
          "total": 2161.679001205365,
          "available": 2161.679001205365,
          "locked": 0.0,
          "blocked": 0.0,
          "avg_price": 0.0
        },
        "ETH": {
          "total": -0.41496350000000076,
          "available": -0.41496350000000076,
          "blocked": 0.0,
          "avg_price": 4461.7130319849075
        }
      },
      "peak_nav": null,
      "volume": 0.0,
      "realised_pnl": 0.0,
      "unrealised_pnl": 0.0,
      "24hr_nav": 333.33,
      "daily_drawdown": 0.0,
      "aggregates": {
        "realised_pnl": -16.899155948068568,
        "volume": 24467.056476149028,
        "rebate": 0.0,
        "total_fees": 0.0,
        "volume_fees": 2.258203156694801
      }
    }
  ],
  "timestamp": "2025-09-20T04:22:37.433546+00:00"
}
---------------------------------------

--- Portfolio State Updated ---
  - okx: USDT: -1428.44, ETH: 0.3950
  - bybit: USDT: 223.71, ETH: 0.0200
  - kucoinspot: USDT